In [ ]:
from pathlib import Path
import whisper
from tqdm import tqdm
import cv2
import numpy as np
import torch
import base64
import requests
import json
import time

def encode_image_b64(img_path: Path) -> str:
    return base64.b64encode(img_path.read_bytes()).decode("utf-8")

In [ ]:
audio_dir = Path("../dataset/clips_audio")
video_dir = Path("../dataset/clips_video")

audio_files = sorted([p.stem for p in audio_dir.glob("*.wav")])
video_files = sorted([p.stem for p in video_dir.glob("*.mp4")])

audio_set = set(audio_files)
video_set = set(video_files)

print(f"Audio files: {len(audio_set)}")
print(f"Video files: {len(video_set)}")

missing_videos = audio_set - video_set
missing_audio = video_set - audio_set

if not missing_videos and not missing_audio:
    print("✅ All filenames match perfectly.")
else:
    print("❌ Mismatches found:")
    if missing_videos:
        print("Missing videos for:", sorted(missing_videos))
    if missing_audio:
        print("Missing audio for:", sorted(missing_audio))


In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using device:", device)
model = whisper.load_model("medium", device="cpu")

output_dir = Path("../dataset/dgo")
output_dir.mkdir(parents=True, exist_ok=True)

for wav_path in tqdm(list(audio_dir.glob("*.wav"))):
    base = wav_path.stem
    out_file = output_dir / f"{base}_transcript.txt"
    
    if out_file.exists():
        continue  # skip already processed
    
    result = model.transcribe(str(wav_path))
    
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(result["text"])


In [ ]:
def extract_frames(video_path, output_dir, n_frames=10):
    cap = cv2.VideoCapture(str(video_path))
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return
    
    frame_indices = np.linspace(
        0, total_frames - 1, n_frames, dtype=int
    )
    
    base = video_path.stem
    saved = 0
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        
        out_path = output_dir / f"{base}_{saved+1}.jpg"
        cv2.imwrite(str(out_path), frame)
        saved += 1
    
    cap.release()

for video_path in tqdm(list(video_dir.glob("*.mp4"))):
    extract_frames(video_path, output_dir, n_frames=10)


### Prompting

In [ ]:
base_prompt = """**Background** You are analyzing a sequence of images accompanied by audio captions and speech transcriptions to identify specific autism-related behaviors. This analysis is part of a clinical or research assessment where accurate behavioral identification is critical for diagnosis, intervention planning, or data collection purposes.\n\n
**Task** Analyze the provided image sequence along with the speech transcription to identify which, if any, of the following nine specific autism-related behavior categories are present:\n
- S1: "Absence or Avoidance of Eye Contact" [A consistent lack of direct gaze or active aversion to maintaining eye contact during social interactions, often limiting nonverbal engagement.]\n
- S2: "Aggressive Behavior" [Physical acts of hostility directed outward toward other people or property, often triggered by communication frustration, disrupted routines, or sensory overwhelm.]\n
- S3: "Hyper or Hyporeactivity to Sensory Input" [An atypical response to sensory stimuli, manifesting as either extreme sensitivity (hyper) or apparent indifference (hypo) to aspects of the environment like sound, light, texture, or temperature.]\n
- S4: "Non-Responsiveness to Verbal Interaction" [A failure to orient towards or acknowledge speech—such as not turning when one's name is called - despite having functional hearing.]\n
- S5: "Non-Typical Language" [The use of idiosyncratic speech patterns, including repetitive phrases (echolalia), made-up words (neologisms), or unusual pitch, intonation, and rhythm.]\n
- S6: "Object Lining-Up" [The compulsive arrangement of toys or items into strict linear formations or patterns, often accompanied by visible distress if the order is disturbed.]\n
- S7: "Self-Hitting or Self-Injurious Behavior" [Repetitive actions that inflict physical harm on oneself, such as head-banging, biting, or skin-picking, often serving as a mechanism for emotional or sensory regulation.]\n
- S8: "Self-Spinning or Spinning Objects" [The repetitive rotation of one's own body or the persistent spinning of external items (e.g., wheels, coins), utilized for vestibular or visual sensory stimulation.]\n
- S9: "Upper Limb Stereotypies" [Repetitive, non-functional motor movements involving the hands or arms, such as hand-flapping, finger-flicking, or waving, commonly referred to as "stimming."]\n
- S10: "Background" [If none of these behaviors are detected, then it is Background]\n\n
**Objective** Provide accurate, evidence-based identification of autism-related behaviors from multimodal inputs (visual and auditory) to support clinical assessment and behavioral documentation.\n\n
**Knowledge** The assistant should understand that:\n
- Multiple behaviors may be present simultaneously in a single sequence\n
- Visual cues in images (body language, facial expressions, object interactions, movement patterns) must be carefully examined\n
- Audio captions and speech transcriptions provide contextual information but may contain transcription errors (as evidenced by "artistic chartistic chart" likely being "autistic characteristics")\n
- Each behavior category has distinct observable markers that differentiate it from others\n
- The absence of clear behavioral indicators should result in a "Background" classification\n
- Behavioral identification must be based on observable evidence from the provided materials, not assumptions\n
**Constraints**\n
- Strictly follow the output format provided at the end\n
- Provide confidence scores or probability assessments\n
- Do not include descriptions of what was observed\n
- Do not suggest additional assessments or recommendations\n
- Do not speculate about behaviors not clearly observable in the provided materials\n
- When uncertain between two similar categories, prioritize the most specific observable behavior\n
- Maintain clinical objectivity and avoid subjective interpretation\n
You must respond using ONLY valid JSON format with the following keys:\n"
        {\n
          "\"signals_present\": [list of strings, e.g., \"S2\", \"S3\"],\n"
          "\"summary\": (string) A narrative summary of the scene,\n"
          "\"visual_details\": (list of strings) Key visual elements noticed,\n"
          "\"confidence\": <float between 0.0 and 1.0>\n"
        }\n\n"""

In [ ]:
sample_lst = list(audio_set)
dir_name = Path("../dataset/dgo")
dataset = {}
for sample in sample_lst:
    
    # Transcript
    transcript_path = dir_name / f"{sample}_transcript.txt"
    if transcript_path.exists():     
        transcript = transcript_path.read_text(encoding="utf-8").strip()
    else:
        transcript  = ""
    
    # Images (max 10)
    image_b64_list = []
    for i in range(1, 11):
        img_path = dir_name / f"{sample}_{i}.jpg"
        if img_path.exists():
            image_b64_list.append(encode_image_b64(img_path))
    
    full_prompt_text = (
        f"{base_prompt}.\n"
        
        f"Transcript context (may be truncated):\n{transcript}"
    )

    # Generate API Payload
    dataset[sample] = {
        "model": "gemma3:27b-it-fp16",
        "stream": False,
        "format": "json",
        "messages": [
            {
                "role": "user",
                "content": full_prompt_text
            },
            {
                "role": "user", 
                "content": "",
                "images": image_b64_list
            }
        ]
    }

In [ ]:
import os

# --- Configuration ---
API_URL = "https://gs1.cht77.com/api/chat"
HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}
OUTPUT_FILE = "../output/llm_output.jsonl"

# --- Helper Function to Save Data ---
def save_result_to_disk(sample_id, data):
    """Appends a single result to the JSONL file immediately."""
    record = {"sample_id": sample_id, **data}
    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record) + "\n")

# --- Main Processing Loop ---
print(f"Starting processing of {len(dataset)} samples...")

count = 0

processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            try:
                processed_ids.add(json.loads(line)['sample_id'])
            except: pass

# Iterate through every sample in your dataset dictionary
for sample_id, payload in dataset.items():

    if sample_id in processed_ids:
        continue

    if count > 0 and count % 30 == 0:
        print(f"\n⏳ Pausing for 30 seconds to respect rate limits... (Processed {count} so far)")
        time.sleep(30)
        print("▶️ Resuming...\n")

    count += 1
    print(f"[{count}] Processing {sample_id}...", end=" ", flush=True)
    
    try:
        # 1. Send the API Request
        response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=60) # Added timeout
        response.raise_for_status()
        
        # 2. Extract Response Data
        api_result = response.json()
        
        # 3. Parse Inner Content
        # We need to get the string content and convert it to a Dict
        if 'message' in api_result:
            model_content_str = api_result['message']['content']
            
            # Clean up potential markdown code blocks if the model adds them (e.g. ```json ... ```)
            if model_content_str.startswith("```"):
                model_content_str = model_content_str.strip("`").replace("json", "").strip()
            
            # Convert string to Python Dictionary
            structured_data = json.loads(model_content_str)
            
            # 4. Save to Disk
            save_result_to_disk(sample_id, structured_data)
            print("✅ Success")
            
        else:
            print(f"⚠️ Unexpected API structure: {api_result.keys()}")

    except requests.exceptions.RequestException as e:
        print(f"❌ Network Error: {e}")
        # Optional: Log the ID to a 'failed.txt' file to retry later
        
    except json.JSONDecodeError as e:
        print(f"❌ JSON Parse Error. Raw content was: {model_content_str[:50]}...")
        
    except Exception as e:
        print(f"❌ Unknown Error: {e}")

    # Optional: Brief sleep to be nice to the API server
    # time.sleep(0.5) 

print(f"\nProcessing complete. Results saved to {OUTPUT_FILE}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

def fix_signals(val):
    # Check for NaN/None
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ['S10']
    # Check for empty list
    if isinstance(val, list) and len(val) == 0:
        return ['S10']
    return val

ground_truth_df = pd.read_csv("../dataset/csvs/dataset.csv")
llm_df = pd.read_json("../output/llm_output.jsonl", lines=True)
reference_df = pd.read_csv("../dataset/csvs/reference_signal.csv")
llm_df['signals_present'] = llm_df['signals_present'].apply(fix_signals)

In [ ]:
# Filter rows where 'S10' is inside the list
s10_rows = llm_df[llm_df['signals_present'].apply(lambda x: 'S10' in x if isinstance(x, list) else False)]

# View the result
print(s10_rows)

In [ ]:
desc_to_code = dict(zip(reference_df['Description'], reference_df['Signal']))
ground_truth_df.columns = [col.strip() for col in ground_truth_df.columns]
ground_truth_df = ground_truth_df.rename(columns=desc_to_code)
ground_truth_df.rename(columns={'Video_ID':'sample_id'}, inplace=True)
display(ground_truth_df)

all_classes = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10']
mlb = MultiLabelBinarizer(classes=all_classes)

llm_encoded = mlb.fit_transform(llm_df['signals_present'])
llm_binary_df = pd.DataFrame(llm_encoded, columns=mlb.classes_, index=llm_df['sample_id'])
llm_binary_df = llm_binary_df.reset_index()

# Check the result
display(llm_binary_df)

In [ ]:
gt_indexed = ground_truth_df.set_index('sample_id')
llm_indexed = llm_binary_df.set_index('sample_id')

# 2. Force the columns to be in the exact same order
# (e.g., S1, S2, S3... S10) to prevent S1 comparing against S2
target_columns = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10']
gt_indexed = gt_indexed[target_columns]
llm_indexed = llm_indexed[target_columns]

# 3. Align them (The Magic Step)
# join='inner': Drops any IDs that are missing in either file (e.g., failed API calls)
# axis=0: Aligns based on the Index (sample_id)
gt_aligned, llm_aligned = gt_indexed.align(llm_indexed, join='inner', axis=0)

# 4. Verification
print(f"Matched {len(gt_aligned)} samples.")
print(f"GT Shape: {gt_aligned.shape}")
print(f"LLM Shape: {llm_aligned.shape}")

In [ ]:
metrics_list = []

for signal in all_classes:
    # Get the actual (Ground Truth) and predicted (LLM) columns
    y_true = gt_aligned[signal]
    y_pred = llm_aligned[signal]
    
    # Calculate TP, FP, TN, FN
    # We use boolean logic to separate them
    TP = ((y_true == 1) & (y_pred == 1)).sum()
    FP = ((y_true == 0) & (y_pred == 1)).sum()
    TN = ((y_true == 0) & (y_pred == 0)).sum()
    FN = ((y_true == 1) & (y_pred == 0)).sum()
    
    # Total samples
    N = TP + FP + TN + FN
    
    # Avoid division by zero
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    
    # F1 Score
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # Other metrics from your doc
    accuracy = (TP + TN) / N
    fpr = 1 - specificity
    fnr = 1 - recall
    prevalence = (TP + FN) / N
    pred_pos_rate = (TP + FP) / N
    
    metrics_list.append({
        'Behavior': signal,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'Specificity': specificity,
        'F1': f1,
        'FPR': fpr,
        'FNR': fnr,
        'Prevalence': prevalence,
        'Pred Pos Rate': pred_pos_rate,
        'TP': TP,
        'FP': FP,
        'TN': TN,
        'FN': FN
    })

# 3. Create the Per-Label DataFrame
metrics_df = pd.DataFrame(metrics_list)

# Map S1-S10 back to descriptions for readability (Optional)
# metrics_df['Behavior'] = metrics_df['Behavior'].map(desc_mapping) # if you have the dict

# Display Per-Label Table
print("--- Per-Label Performance Metrics ---")
print(metrics_df.round(3))


# --- 4. Calculate Overall Micro-Averages ---

# Sum the raw counts across all labels
sum_tp = metrics_df['TP'].sum()
sum_fp = metrics_df['FP'].sum()
sum_tn = metrics_df['TN'].sum()
sum_fn = metrics_df['FN'].sum()

# Calculate Micro Metrics based on sums
micro_precision = sum_tp / (sum_tp + sum_fp) if (sum_tp + sum_fp) > 0 else 0
micro_recall = sum_tp / (sum_tp + sum_fn) if (sum_tp + sum_fn) > 0 else 0
micro_f1 = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0
micro_accuracy = (sum_tp + sum_tn) / (sum_tp + sum_tn + sum_fp + sum_fn)

overall_metrics = {
    'Metric': ['Micro-Precision', 'Micro-Recall', 'Micro-F1', 'Micro-Accuracy', 'Σ TP', 'Σ FP', 'Σ TN', 'Σ FN'],
    'Value': [micro_precision, micro_recall, micro_f1, micro_accuracy, sum_tp, sum_fp, sum_tn, sum_fn]
}

overall_df = pd.DataFrame(overall_metrics)

print("\n--- Overall Micro-Averages ---")
print(overall_df.round(4))